# EDA maestro de precios y contexto agrícola

Este notebook consume el workbook generado por `scripts.build_master_price_workbook`.

Objetivos principales:
- comparar las series diarias de SNIIM, Walmart y Chedraui
- usar Avance Agrícola como contexto mensual principal
- dejar Cierre Agrícola como referencia anual secundaria


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

WORKBOOK_PATH = Path(r"C:\Users\Dell-G3\Documents\Jupyter-projects\Others\agro-precios\data\analysis\master_price_workbook.xlsx")
COMPARE_SHEET = "compare_daily_wide"
COVERAGE_SHEET = "coverage"
AVANCE_MONTHLY_SHEET = "avance_monthly_stats"
AVANCE_ENTITY_SHEET = "avance_entity_monthly"
CIERRE_SHEET = "cierre_annual_stats"

PRODUCTO = "aguacate"
FECHA_INICIO = None
FECHA_FIN = None
METRICA_AVANCE = "avance_total_produccion"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)


## Carga y validación

Esta sección carga las hojas necesarias, valida que existan y muestra un resumen base del rango temporal y la cobertura.


In [ ]:
required_sheets = [COMPARE_SHEET, COVERAGE_SHEET, AVANCE_MONTHLY_SHEET, AVANCE_ENTITY_SHEET]
excel_file = pd.ExcelFile(WORKBOOK_PATH)
missing_sheets = [sheet for sheet in required_sheets if sheet not in excel_file.sheet_names]
if missing_sheets:
    raise ValueError(f"Faltan hojas requeridas en el workbook: {missing_sheets}")

compare_df = pd.read_excel(WORKBOOK_PATH, sheet_name=COMPARE_SHEET)
coverage_df = pd.read_excel(WORKBOOK_PATH, sheet_name=COVERAGE_SHEET)
avance_monthly_df = pd.read_excel(WORKBOOK_PATH, sheet_name=AVANCE_MONTHLY_SHEET)
avance_entity_df = pd.read_excel(WORKBOOK_PATH, sheet_name=AVANCE_ENTITY_SHEET)
cierre_df = pd.read_excel(WORKBOOK_PATH, sheet_name=CIERRE_SHEET) if CIERRE_SHEET in excel_file.sheet_names else pd.DataFrame()

compare_df["run_date"] = pd.to_datetime(compare_df["run_date"])
coverage_df["run_date"] = pd.to_datetime(coverage_df["run_date"])
if "avance_cutoff_date" in compare_df.columns:
    compare_df["avance_cutoff_date"] = pd.to_datetime(compare_df["avance_cutoff_date"], errors="coerce")
if "avance_cutoff_date" in avance_monthly_df.columns:
    avance_monthly_df["avance_cutoff_date"] = pd.to_datetime(avance_monthly_df["avance_cutoff_date"], errors="coerce")

if FECHA_INICIO is not None:
    compare_df = compare_df[compare_df["run_date"] >= pd.Timestamp(FECHA_INICIO)].copy()
    coverage_df = coverage_df[coverage_df["run_date"] >= pd.Timestamp(FECHA_INICIO)].copy()
if FECHA_FIN is not None:
    compare_df = compare_df[compare_df["run_date"] <= pd.Timestamp(FECHA_FIN)].copy()
    coverage_df = coverage_df[coverage_df["run_date"] <= pd.Timestamp(FECHA_FIN)].copy()

productos_disponibles = sorted(compare_df["canonical_product"].dropna().astype(str).unique().tolist())
print(f"Filas en compare_daily_wide: {len(compare_df):,}")
print(f"Filas en coverage: {len(coverage_df):,}")
print(f"Filas en avance_monthly_stats: {len(avance_monthly_df):,}")
print(f"Filas en avance_entity_monthly: {len(avance_entity_df):,}")
print(f"Rango de fechas diario: {compare_df['run_date'].min().date()} -> {compare_df['run_date'].max().date()}")
print(f"Productos disponibles: {len(productos_disponibles)}")
display(pd.DataFrame({"producto": productos_disponibles}).head(20))

coverage_share = coverage_df[["has_sniim", "has_walmart", "has_chedraui", "has_avance", "has_cierre"]].mean(numeric_only=True)
display(coverage_share.rename("proporcion_cobertura").to_frame())


## Vista diaria de precios

Se grafican los precios diarios de las fuentes de mercado junto con una métrica mensual de Avance Agrícola como contexto en eje secundario.


In [ ]:
avance_monthly_df


In [ ]:
# Filtramos el producto seleccionado y ordenamos por fecha.
product_df = compare_df[compare_df["canonical_product"].astype(str).str.casefold() == PRODUCTO.casefold()].copy()
product_df = product_df.sort_values("run_date")
if product_df.empty:
    raise ValueError(f"No hay datos para el producto seleccionado: {PRODUCTO}")

avance_product_df = avance_monthly_df[
    avance_monthly_df["canonical_product"].astype(str).str.casefold() == PRODUCTO.casefold()
].copy().sort_values(["query_year", "query_month"])

metricas_avance_validas = {
    "avance_total_produccion": "Producción mensual",
    "avance_total_superficie_cosechada_ha": "Superficie cosechada mensual",
    "avance_yield_weighted_udm_ha": "Rendimiento mensual",
}
if METRICA_AVANCE not in metricas_avance_validas:
    raise ValueError(f"METRICA_AVANCE debe ser una de: {list(metricas_avance_validas)}")

fig, ax_precio = plt.subplots(figsize=(14, 6))
series_precio = [
    ("sniim_daily_mean_mxn", "SNIIM"),
    ("walmart_comparison_mxn", "Walmart"),
    ("chedraui_comparison_mxn", "Chedraui"),
]
for column, label in series_precio:
    if column in product_df.columns and product_df[column].notna().any():
        ax_precio.plot(product_df["run_date"], product_df[column], marker="o", linewidth=2, label=label)

ax_precio.set_title(f"Precios diarios y contexto agrícola mensual: {PRODUCTO}")
ax_precio.set_xlabel("Fecha")
ax_precio.set_ylabel("Precio (MXN)")

ax_avance = ax_precio.twinx()
if not avance_product_df.empty:
    x_avance = pd.to_datetime(
        avance_product_df["query_year"].astype(int).astype(str)
        + "-"
        + avance_product_df["query_month"].astype(int).astype(str).str.zfill(2)
        + "-01"
    )
    ax_avance.step(
        x_avance,
        avance_product_df[METRICA_AVANCE],
        where="post",
        color="black",
        linewidth=2,
        linestyle="--",
        label=metricas_avance_validas[METRICA_AVANCE],
    )
    ax_avance.set_ylabel(metricas_avance_validas[METRICA_AVANCE])
else:
    ax_avance.set_ylabel("Contexto de Avance no disponible")

handles_1, labels_1 = ax_precio.get_legend_handles_labels()
handles_2, labels_2 = ax_avance.get_legend_handles_labels()
ax_precio.legend(handles_1 + handles_2, labels_1 + labels_2, loc="upper left")
plt.tight_layout()
plt.show()


## Brechas retail vs contexto agrícola

Aquí se calculan las brechas diarias contra SNIIM y se resumen por mes para relacionarlas con producción, superficie cosechada y rendimiento.


In [ ]:
# Calculamos las brechas solo donde exista SNIIM como referencia diaria.
spread_df = product_df[["run_date", "canonical_product", "query_year", "query_month"]].copy()
spread_df["brecha_walmart_vs_sniim"] = product_df["walmart_comparison_mxn"] - product_df["sniim_daily_mean_mxn"]
spread_df["brecha_chedraui_vs_sniim"] = product_df["chedraui_comparison_mxn"] - product_df["sniim_daily_mean_mxn"]
spread_df["mes_clave"] = pd.to_datetime(spread_df["run_date"]).dt.to_period("M").astype(str)
spread_df["mes_fecha"] = pd.to_datetime(spread_df["mes_clave"] + "-01")

spread_monthly = (
    spread_df.groupby(["query_year", "query_month", "mes_clave", "mes_fecha"], dropna=False)
    .agg(
        brecha_promedio_walmart_vs_sniim=("brecha_walmart_vs_sniim", "mean"),
        brecha_promedio_chedraui_vs_sniim=("brecha_chedraui_vs_sniim", "mean"),
        dias_con_datos=("run_date", "count"),
    )
    .reset_index()
)
spread_monthly = spread_monthly.merge(
    avance_product_df[
        [
            "query_year",
            "query_month",
            "avance_total_produccion",
            "avance_total_superficie_cosechada_ha",
            "avance_yield_weighted_udm_ha",
            "avance_siniestrada_share",
        ]
    ],
    on=["query_year", "query_month"],
    how="left",
)

display(spread_monthly)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
axes[0].plot(spread_df["run_date"], spread_df["brecha_walmart_vs_sniim"], label="Walmart - SNIIM", linewidth=2)
axes[0].plot(spread_df["run_date"], spread_df["brecha_chedraui_vs_sniim"], label="Chedraui - SNIIM", linewidth=2)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title(f"Brechas diarias contra SNIIM: {PRODUCTO}")
axes[0].set_ylabel("Brecha (MXN)")
axes[0].legend()

axes[1].bar(spread_monthly["mes_fecha"], spread_monthly["avance_total_produccion"], width=20, label="Producción mensual")
axes[1].plot(spread_monthly["mes_fecha"], spread_monthly["brecha_promedio_walmart_vs_sniim"], marker="o", label="Brecha promedio Walmart - SNIIM")
axes[1].plot(spread_monthly["mes_fecha"], spread_monthly["brecha_promedio_chedraui_vs_sniim"], marker="o", label="Brecha promedio Chedraui - SNIIM")
axes[1].set_title("Brechas promedio mensuales vs producción de Avance")
axes[1].set_xlabel("Mes")
axes[1].set_ylabel("Valor")
axes[1].legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Contexto operativo de Avance

Se muestran el último corte disponible, los cambios mes contra mes y el detalle por entidad para entender la base agrícola detrás de las series de precios.


In [ ]:
avance_product_df = avance_product_df.sort_values(["query_year", "query_month"]).copy()
if avance_product_df.empty:
    print(f"No hay contexto de Avance para {PRODUCTO}.")
else:
    avance_product_df["produccion_variacion_abs"] = avance_product_df["avance_total_produccion"].diff()
    avance_product_df["cosechada_variacion_abs"] = avance_product_df["avance_total_superficie_cosechada_ha"].diff()
    avance_product_df["siniestrada_variacion_abs"] = avance_product_df["avance_total_superficie_siniestrada_ha"].diff()
    avance_product_df["rendimiento_variacion_abs"] = avance_product_df["avance_yield_weighted_udm_ha"].diff()

    ultimo_corte = avance_product_df.sort_values("avance_cutoff_date").tail(1)
    print("Último corte mensual disponible")
    display(ultimo_corte)

    print("Cambios mes contra mes")
    display(
        avance_product_df[
            [
                "avance_month_key",
                "avance_total_produccion",
                "produccion_variacion_abs",
                "avance_total_superficie_cosechada_ha",
                "cosechada_variacion_abs",
                "avance_total_superficie_siniestrada_ha",
                "siniestrada_variacion_abs",
                "avance_yield_weighted_udm_ha",
                "rendimiento_variacion_abs",
            ]
        ]
    )

    latest_year = int(ultimo_corte.iloc[0]["query_year"])
    latest_month = int(ultimo_corte.iloc[0]["query_month"])
    entity_latest = avance_entity_df[
        (avance_entity_df["canonical_product"].astype(str).str.casefold() == PRODUCTO.casefold())
        & (pd.to_numeric(avance_entity_df["query_year"], errors="coerce") == latest_year)
        & (pd.to_numeric(avance_entity_df["query_month"], errors="coerce") == latest_month)
    ].copy()
    entity_latest = entity_latest.sort_values("produccion", ascending=False)
    print("Detalle por entidad del último mes disponible")
    display(entity_latest.head(15))

    top_entities = entity_latest.head(10)
    if not top_entities.empty:
        ax = top_entities.plot(
            kind="bar",
            x="entidad",
            y="produccion",
            figsize=(12, 5),
            title=f"Top entidades por producción en Avance: {PRODUCTO}",
            legend=False,
        )
        ax.set_xlabel("Entidad")
        ax.set_ylabel("Producción")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()


## Cobertura y calidad de datos

Esta sección resume la presencia de cada fuente y detecta meses donde existan precios sin contexto de Avance o contexto sin suficientes precios diarios.


In [ ]:
coverage_product_df = coverage_df[
    coverage_df["canonical_product"].astype(str).str.casefold() == PRODUCTO.casefold()
].copy().sort_values("run_date")

coverage_summary = coverage_df[["has_sniim", "has_walmart", "has_chedraui", "has_avance", "has_cierre"]].mean(numeric_only=True).sort_values(ascending=False)
ax = coverage_summary.plot(kind="bar", figsize=(9, 4), title="Proporción de cobertura por fuente")
ax.set_xlabel("Fuente")
ax.set_ylabel("Proporción de filas con datos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

faltantes_mensuales = (
    coverage_product_df.assign(mes_clave=coverage_product_df["run_date"].dt.to_period("M").astype(str))
    .groupby("mes_clave", dropna=False)
    .agg(
        dias_total=("run_date", "count"),
        dias_con_avance=("has_avance", "sum"),
        dias_con_sniim=("has_sniim", "sum"),
        dias_con_walmart=("has_walmart", "sum"),
        dias_con_chedraui=("has_chedraui", "sum"),
    )
    .reset_index()
)
faltantes_mensuales["mes_con_precios_sin_avance"] = (
    (faltantes_mensuales[["dias_con_sniim", "dias_con_walmart", "dias_con_chedraui"]].sum(axis=1) > 0)
    & (faltantes_mensuales["dias_con_avance"] == 0)
)
display(faltantes_mensuales)


## Contexto anual secundario

Cierre Agrícola queda como referencia anual opcional. Si la hoja existe y tiene datos para el producto, se muestra aquí sin mezclarla con la visualización principal diaria.


In [ ]:
if cierre_df.empty:
    print("No se cargó contexto anual de Cierre Agrícola en este workbook.")
else:
    cierre_product_df = cierre_df[cierre_df["canonical_product"].astype(str).str.casefold() == PRODUCTO.casefold()].copy()
    if cierre_product_df.empty:
        print(f"No hay datos de Cierre Agrícola para {PRODUCTO}.")
    else:
        print("Referencia anual de Cierre Agrícola")
        display(cierre_product_df)
        ax = cierre_product_df.plot(
            x="query_year",
            y="cierre_annual_weighted_pmr_mxn_udm",
            marker="o",
            figsize=(10, 4),
            title=f"PMR anual ponderado de Cierre Agrícola: {PRODUCTO}",
            legend=False,
        )
        ax.set_xlabel("Año")
        ax.set_ylabel("PMR ponderado (MXN / udm)")
        plt.tight_layout()
        plt.show()
